# Практика: моржовый оператор и match-case

На этом занятии обсудим:

- моржовый оператор `:=` — присваивание прямо внутри выражения;
- конструкцию `match`-`case` (структурное сопоставление с образцом,
  Python 3.10+): последовательности, литералы, `*`-захват, wildcard `_`,
  or-образцы, `as`-образцы, условия-«стражи» (guards).

Конструкцию `match`-`case` будем разбирать **итеративно**: шаг за шагом
напишем разбор команд для текстового квеста. За основу взят официальный
туториал [PEP 636 — Structural Pattern Matching: Tutorial](https://peps.python.org/pep-0636/).

## 1. Моржовый оператор `:=`

Обычный оператор `=` — это **инструкция**: она связывает имя со значением,
но сама значения не имеет, поэтому использовать ее внутри выражения нельзя.

В Python 3.8 появился **оператор присваивания-выражения** `:=` (PEP 572).
За характерный вид (`:=` похоже на глаза и бивни) его называют **моржовым**.

```python
имя := выражение
```

Он делает сразу две вещи:

1. вычисляет выражение и связывает результат с именем;
2. **возвращает** это значение — как результат всего выражения `имя := выражение`.

Поэтому его можно встраивать в другие выражения: в условия `if` и `while`,
аргументы функций, включения списков.

In [ ]:
print(city_name := "Moscow")  # присваивание внутри выражения
print(city_name)              # имя осталось связанным

На «верхнем уровне» инструкции моржовый оператор без скобок запрещен — для
обычного присваивания есть `=`:

In [ ]:
number := 10

Если очень хочется, нужны скобки (хотя так писать не стоит — для этого есть `=`):

In [ ]:
(number := 10)
print(number)

### Зачем он нужен: не вычислять одно и то же дважды

Типичная ситуация — значение нужно и в условии, и в теле `if`. Без `:=`
приходится либо вычислять его дважды, либо заводить переменную отдельной
строкой перед `if`.

In [ ]:
text = "  python is fun  "

if len(text.strip()) > 5:
    print(f"Long text: {len(text.strip())} characters")  # strip() и len() вызваны дважды

Второй вариант — вынести значение в переменную отдельной строкой перед `if`.
Вычисление теперь одно, но появилась лишняя строка, а имя `stripped_length`
«повисло» перед условием, хотя нужно только ему:

In [ ]:
text = "  python is fun  "
stripped_length = len(text.strip())

if stripped_length > 5:
    print(f"Long text: {stripped_length} characters")

Моржовый оператор позволяет совместить вычисление, сохранение и проверку:

In [ ]:
text = "  python is fun  "

if (stripped_length := len(text.strip())) > 5:
    print(f"Long text: {stripped_length} characters")

Еще один пример — метод `find()`, который возвращает `-1`, если подстроку
не нашли, и позицию подстроки иначе. Позиция нужна и для проверки, и для
дальнейшей работы:

In [ ]:
email = "student@example.com"

if (at_position := email.find("@")) != -1:
    login = email[:at_position]
    domain = email[at_position + 1:]
    print(login, domain)  # student example.com

else:
    print("It is not an email")

### Моржовый оператор в цикле `while`

Классический пример — чтение ввода пользователя. Без `:=` вызов `input()`
приходится писать **дважды**: до цикла и в конце тела цикла.

> Если запускаете ячейки в VS Code, поле для ввода появится в верхней части
> окна. Чтобы остановить цикл, введите пустую строку.

In [ ]:
user_input = input("Enter a word (empty line to stop): ")

while user_input != "":
    print(f"You entered: {user_input}")
    user_input = input("Enter a word (empty line to stop): ")

С моржовым оператором чтение и проверка объединяются в одну строку, а
дублирование исчезает:

In [ ]:
while (user_input := input("Enter a word (empty line to stop): ")) != "":
    print(f"You entered: {user_input}")

А еще вспомним про истинность строк: пустая строка ложна, поэтому сравнение
с `""` можно убрать. Когда `:=` — **целиком** условие, скобки вокруг него не
обязательны:

In [ ]:
while user_input := input("Enter a word (empty line to stop): "):
    print(f"You entered: {user_input}")

### Моржовый оператор во включениях списков

Допустим, нужно очистить слова от пробелов, выбросить пустые и привести
остальные к верхнему регистру. Результат `strip()` нужен дважды: в фильтре
и в выражении элемента.

In [ ]:
words = ["  Python ", "", "  ", "match", " case "]

cleaned_words = [
    stripped_word.upper()
    for word in words
    if (stripped_word := word.strip())
]
print(cleaned_words)  # ['PYTHON', 'MATCH', 'CASE']

Обратите внимание: имя, связанное через `:=` внутри включения, **остается
доступным после него** (в отличие от переменной цикла `word`).

In [ ]:
print(stripped_word)  # case — последнее значение, которое получило имя

### Нюансы и предостережения

- **Приоритет у `:=` самый низкий**, поэтому в составных выражениях его
  нужно брать в скобки. Классическая ошибка — забыть скобки в сравнении:

In [ ]:
if text_length := len("python") > 5:  # без скобок
    print(text_length)                # True, а не 6!

  Здесь `:=` применился ко **всему** выражению `len("python") > 5`, поэтому
  в `text_length` попал результат сравнения. Правильно: `(text_length := len("python")) > 5`.

- Слева от `:=` может стоять только **простое имя**. Присваивать элементам
  списка (`items[0] := 1`), атрибутам или распаковывать (`first, second := ...`)
  так нельзя.
- `:=` не заменяет `=`. Если значение нужно только сохранить, пишите
  обычное присваивание отдельной строкой.
- Не злоупотребляйте. Если из-за `:=` строка стала трудночитаемой — лучше
  разбить ее на несколько строк.

### Задание 1: английские слова из строки

**Условие.**

Реализуйте функцию `extract_english_words(text)`, которая принимает строку
и возвращает **строку**. Входная строка устроена так:

- в ней есть слова — любые последовательности символов, разделенные пробелами;
- слова могут быть разделены произвольным числом пробелов;
- в начале и в конце строки могут быть пробелы.

Функция должна:

1. извлечь из строки слова;
2. оставить только те слова, которые состоят **исключительно из английских
   букв** (латиница `a-z`, `A-Z`);
3. привести оставшиеся слова к нижнему регистру;
4. склеить их в одну строку через **один** пробел.

Если подходящих слов нет, функция возвращает пустую строку.

**Подсказка.** Метод `isalpha()` возвращает `True` и для русских букв, поэтому
одного его недостаточно — подумайте, чем его дополнить.

In [ ]:
def extract_english_words(text: str) -> str:
    # ваш код
    return text

### Проверка

In [ ]:
assert extract_english_words("") == ""
assert extract_english_words("     ") == ""
assert extract_english_words("Hello") == "hello"
assert extract_english_words("  Hello   WORLD  ") == "hello world"
assert extract_english_words("Python3 is fun 42") == "is fun"
assert extract_english_words("Привет world мир") == "world"
assert extract_english_words("don't stop, GO") == "go"
assert extract_english_words("café Cafe") == "cafe"
assert extract_english_words("123 !!! привет") == ""

print("All tests passed")

## 2. Конструкция `match`-`case`

В Python 3.10 появилась конструкция `match`-`case` — **структурное
сопоставление с образцом** (structural pattern matching). Ее часто
называют «`switch` из Python», но это сильное упрощение:

- `switch` в других языках только сравнивает значение с константами;
- `match` проверяет **структуру** значения (это последовательность из двух
  элементов? первый элемент — строка `"go"`?) и одновременно **достает
  из него части** в переменные.

### Синтаксис

```python
match выражение:
    case образец_1:
        действия
    case образец_2 if условие:
        действия
    case _:
        действия
```

Правила работы:

1. выражение после `match` (его называют **субъектом**) вычисляется **один раз**;
2. образцы `case` проверяются **сверху вниз**;
3. выполняется тело **первого** подошедшего `case`, остальные пропускаются
   (никакого «проваливания» как в `switch` в C);
4. если ни один образец не подошел (и нет `case _`) — не происходит
   ничего, и это **не ошибка**.

`match` и `case` — «мягкие» ключевые слова: они имеют особое значение только
в этой конструкции, а в остальных местах остаются обычными именами.

Самый простой образец — **литерал**. Тогда `match` работает как привычный
`switch`:

In [ ]:
http_status = 404

match http_status:
    case 200:
        print("OK")

    case 404:
        print("Not found")

    case 500:
        print("Server error")

    case _:
        print("Unknown status")

Знак `_` — **wildcard**: образец, который подходит вообще к чему угодно.
Он играет роль `default` / `else`.

Но настоящая сила `match` проявляется не здесь. Давайте убедимся в этом на
практике — будем писать **текстовый квест**.

## 3. Пишем текстовый квест

Игрок вводит команды вида:

```
look
get sword
go north
drop sword shield
```

Наша задача — разобрать команду и понять, что игрок хочет сделать. Сначала
превратим строку в список слов с помощью `str.split()`:

In [ ]:
print("get sword".split())  # ['get', 'sword']
print("look".split())       # ['look']
print("".split())           # []

Теперь список слов можно сопоставлять с образцами. Чтобы не вводить команды
руками на каждом шаге, подготовим набор тестовых команд и будем
прогонять их через очередную версию `match`:

In [ ]:
test_commands = [
    "look",
    "quit",
    "exit",
    "get sword",
    "pick up sword",
    "go north",
    "go up",
    "drop sword shield",
    "dance",
    "",
]

### Шаг 1. Образец-последовательность

Образец `[action, target]` — это **образец-последовательность**
(sequence pattern). Он подходит к любой последовательности **ровно из двух
элементов**, а имена `action` и `target` — это **образцы-захваты**
(capture patterns): при совпадении они связываются с соответствующими
элементами.

Обратите внимание: что не подошло, то молча пропускается.

In [ ]:
for command in test_commands:
    print(f"> {command!r}")

    match command.split():
        case [action, target]:
            print(f"Action: {action}, target: {target}")

### Шаг 2. Несколько образцов

Добавим образец для команд из одного слова. Образцы проверяются сверху
вниз, выполняется первый подошедший.

In [ ]:
for command in test_commands:
    print(f"> {command!r}")

    match command.split():
        case [action]:
            print(f"One-word command: {action}")

        case [action, target]:
            print(f"Two-word command: {action} {target}")

### Шаг 3. Литералы в образцах

Пока что мы реагируем на любые слова одинаково. Научим программу различать
конкретные команды: вместо имени `action` подставим **литерал** — строку.

```python
case ["get", item]
```

означает: «список из двух элементов, **первый равен** `"get"`, второй
запомни в `item`». Литералы сравниваются через `==` (исключение — `None`,
`True` и `False`: они сравниваются через `is`).

In [ ]:
for command in test_commands:
    print(f"> {command!r}")

    match command.split():
        case ["quit"]:
            print("Goodbye!")

        case ["look"]:
            print("You see a dark room.")

        case ["get", item]:
            print(f"You picked up: {item}")

        case ["go", direction]:
            print(f"You go {direction}")

Теперь «настоящие» команды обрабатываются, а остальные — пока игнорируются.

> **Важно: порядок образцов имеет значение.** Если поставить общий образец
> `[action, target]` **раньше** `["get", item]`, то команда `get sword`
> всегда будет попадать в общий образец, а более конкретный станет
> недостижимым. Поэтому конкретные образцы ставят выше общих.

### Шаг 4. Переменное число элементов: `*`

Игрок хочет бросить сразу несколько предметов: `drop sword shield`. Как и
при распаковке (`first, *rest = ...`), в образце-последовательности можно
использовать **`*имя`** — оно «заберет» все оставшиеся элементы в
**список** (возможно, пустой). Звездочка в образце может быть только одна.

In [ ]:
for command in test_commands:
    print(f"> {command!r}")

    match command.split():
        case ["quit"]:
            print("Goodbye!")

        case ["look"]:
            print("You see a dark room.")

        case ["get", item]:
            print(f"You picked up: {item}")

        case ["go", direction]:
            print(f"You go {direction}")

        case ["drop", *items]:
            print(f"You dropped: {items}")

### Шаг 5. Подстраховка: `case _`

Хорошо бы сообщать игроку, что его команда не понята. Для этого нужен
образец, который подходит ко **всему** — wildcard `_`. Он, в отличие от
`action` и `target`, ничего не связывает с именем.

Заодно обработаем пустую команду: `"".split()` дает пустой список, которому
соответствует образец `[]`.

In [ ]:
for command in test_commands:
    print(f"> {command!r}")

    match command.split():
        case ["quit"]:
            print("Goodbye!")

        case ["look"]:
            print("You see a dark room.")

        case ["get", item]:
            print(f"You picked up: {item}")

        case ["go", direction]:
            print(f"You go {direction}")

        case ["drop", *items]:
            print(f"You dropped: {items}")

        case []:
            print("Say something!")

        case _:
            print(f"Sorry, I couldn't understand {command!r}")

`case _` подходит ко всему, поэтому он **обязан стоять последним**: иначе
все образцы после него были бы недостижимы, и Python сообщит об этом ошибкой
еще до запуска:

In [ ]:
match "get sword".split():
    case _:
        print("Anything")

    case ["quit"]:
        print("Goodbye!")

### Шаг 6. Образцы можно вкладывать

Образцы можно **составлять** друг из друга: в образец-последовательность
можно вложить другую последовательность, литералы, `*`-захват и т. д.
Это позволяет за одно сопоставление проверить структуру сложных данных и
достать нужные части:

In [ ]:
record = ["first", (10, 20), "ignored", 1, 2, 3]

match record:
    case ["first", (left, right), _, *rest]:
        print(left, right, rest)  # 10 20 [1, 2, 3]

### Шаг 7. Or-образцы: `|`

Игрок может захотеть выйти не только командой `quit`, но и `exit`, а
поднять предмет — `get sword`, `pick up sword` или `pick sword up`. Для
таких синонимов у образцов есть оператор **`|`** («или»):
образец подходит, если подошла хотя бы одна альтернатива.

In [ ]:
for command in test_commands:
    print(f"> {command!r}")

    match command.split():
        case ["quit"] | ["exit"]:
            print("Goodbye!")

        case ["look"]:
            print("You see a dark room.")

        case ["get", item] | ["pick", "up", item] | ["pick", item, "up"]:
            print(f"You picked up: {item}")

        case ["go", direction]:
            print(f"You go {direction}")

        case ["drop", *items]:
            print(f"You dropped: {items}")

        case []:
            print("Say something!")

        case _:
            print(f"Sorry, I couldn't understand {command!r}")

Правило: **все альтернативы должны связывать одни и те же имена**. Иначе
после совпадения было бы непонятно, есть ли `item` вообще:

In [ ]:
match "look".split():
    case ["get", item] | ["look"]:
        print("Matched")

### Шаг 8. Захват подобразца: `as`

Сейчас команда `go up` (или `go banana`) считается корректной. Ограничим
направления четырьмя сторонами света. Это можно сделать, вложив в образец
or-образец из литералов. Но тогда само слово, которое подошло, пропадет —
а оно нам нужно. Ключевое слово **`as`** связывает подошедшую часть с именем:

```python
("north" | "south" | "east" | "west") as direction
```

In [ ]:
for command in test_commands:
    print(f"> {command!r}")

    match command.split():
        case ["quit"] | ["exit"]:
            print("Goodbye!")

        case ["look"]:
            print("You see a dark room.")

        case ["get", item] | ["pick", "up", item] | ["pick", item, "up"]:
            print(f"You picked up: {item}")

        case ["go", ("north" | "south" | "east" | "west") as direction]:
            print(f"You go {direction}")

        case ["drop", *items]:
            print(f"You dropped: {items}")

        case []:
            print("Say something!")

        case _:
            print(f"Sorry, I couldn't understand {command!r}")

Теперь `go up` попадает в `case _`. Но сообщение «не понял команду» тут
не совсем честное: команду-то мы поняли, просто идти туда нельзя.

### Шаг 9. Условия в образцах: guards

К образцу можно добавить **условие** (guard) — `if` после образца.
Образец сначала должен подойти, имена связываются, и только потом
вычисляется условие. Если оно ложно, `match` переходит к следующему `case`.

Допустим, из текущей комнаты выходы есть только на север и на восток.
Список выходов меняется по ходу игры, поэтому «зашить» его в образец
(как мы сделали на шаге 8) нельзя — тут как раз нужен guard.

In [ ]:
available_exits = ["north", "east"]

for command in test_commands:
    print(f"> {command!r}")

    match command.split():
        case ["quit"] | ["exit"]:
            print("Goodbye!")

        case ["look"]:
            print("You see a dark room.")

        case ["get", item] | ["pick", "up", item] | ["pick", item, "up"]:
            print(f"You picked up: {item}")

        case ["go", direction] if direction in available_exits:
            print(f"You go {direction}")

        case ["go", _]:
            print("You can't go that way")

        case ["drop", *items]:
            print(f"You dropped: {items}")

        case []:
            print("Say something!")

        case _:
            print(f"Sorry, I couldn't understand {command!r}")

## 4. Задание: пишем игру целиком

Теперь у нашего парсера команд есть все, что нужно. Соберем из него
настоящую игру. Игровой мир уже описан — за него отвечают три функции,
написанные тоже через `match`:

- `get_next_room(room, direction)` — куда ведет выход из комнаты (или `None`,
  если выхода нет). Субъектом `match` здесь служит **кортеж**
  `(комната, направление)`, и образцы сопоставляют его целиком;
- `describe_room(room)` — текстовое описание комнаты;
- `list_items(room, inventory)` — предметы, которые лежат в комнате и еще не
  подобраны игроком.

Изучите их, а потом запустите ячейку и поэкспериментируйте.

In [ ]:
def get_next_room(room: str, direction: str) -> str | None:
    match (room, direction):
        case ("hall", "north"):
            return "library"

        case ("hall", "east"):
            return "garden"

        case ("library", "south") | ("garden", "west"):
            return "hall"

        case _:
            return None


def describe_room(room: str) -> str:
    match room:
        case "hall":
            return "You are in the hall. Exits: north, east."

        case "library":
            return "You are in the library. Exits: south."

        case "garden":
            return "You are in the garden. A locked gate is here. Exits: west."

        case _:
            return "You are nowhere."


def list_items(room: str, inventory: list[str]) -> list[str]:
    match room:
        case "hall":
            all_items = ["lamp"]

        case "library":
            all_items = ["book", "key"]

        case _:
            all_items = []

    return [item for item in all_items if item not in inventory]


print(get_next_room("hall", "north"))  # library
print(get_next_room("hall", "up"))     # None
print(describe_room("garden"))
print(list_items("library", ["book"]))  # ['key']

### Что нужно сделать

Игрок находится в зале (`hall`) с пустым инвентарем. Цель игры — найти в
библиотеке ключ и открыть им калитку в саду. Реализуйте цикл обработки
команд с помощью `match`-`case`, чтобы игра отвечала на следующие команды
(тексты сообщений должны быть именно такими):

| Команда | Что делает |
|---|---|
| `quit` или `exit` | выводит `Goodbye!` и завершает игру |
| `look` | выводит описание текущей комнаты, а затем `Items here: [...]` — доступные в ней предметы |
| `inventory` | выводит `You have: [...]` — список подобранных предметов |
| `go <направление>` | если выход есть — переходит в другую комнату и выводит ее описание, иначе выводит `You can't go that way` |
| `get <предмет>` или `take <предмет>` | если предмет доступен в комнате — кладет его в инвентарь и выводит `You picked up: <предмет>`, иначе выводит `There is no <предмет> here` |
| `open gate` | если игрок в саду (`garden`) и у него есть `key` — выводит `You opened the gate with the key. You win!` и завершает игру, иначе выводит `The gate is locked or too far away` |
| пустая строка | выводит `Say something!` |
| любая другая | выводит `Unknown command: <первое слово команды>` |

**Подсказки.**

- Заготовка цикла ниже уже содержит обработку `quit` / `exit` — продолжайте
  по аналогии. Нужные вам переменные `current_room`, `inventory` и
  `available_items` (предметы, доступные в текущей комнате) уже есть.
- Синонимы `quit` / `exit` и `get` / `take` удобно разобрать **or-образцом**.
- Команда `go` — хороший случай для guard вместе с моржовым оператором: в
  условии нужно **и проверить**, что выход существует, **и получить**
  название комнаты, куда он ведет. А в теле `case` использовать готовое
  значение:

  ```python
  case ["go", direction] if (next_room := get_next_room(current_room, direction)):
      ...
  ```
- Для `get` / `take` и `open gate` понадобятся guard'ы: сначала более
  конкретный `case` с условием, ниже — общий для случая, когда условие
  не выполнилось.
- Пустую команду ловит образец `[]`, а любую непустую — образец
  `[first_word, *_]` (первое слово запоминаем, остальные отбрасываем). Вместе
  они покрывают все возможные списки слов, поэтому `case _` не нужен.
- Помните: **порядок `case` важен** — конкретные образцы выше общих.

Для запуска ячейки в VS Code поле для ввода появится в верхней части окна.

In [ ]:
current_room = "hall"
inventory: list[str] = []

print(describe_room(current_room))

while True:
    command = input("> ")
    available_items = list_items(current_room, inventory)

    match command.split():
        case ["quit"] | ["exit"]:
            print("Goodbye!")
            break

        # ваш код

### Пример игровой сессии

Так должна выглядеть игра, если все реализовано верно (строки с `>` —
ввод игрока):

```
You are in the hall. Exits: north, east.
> look
You are in the hall. Exits: north, east.
Items here: ['lamp']
> go up
You can't go that way
> go north
You are in the library. Exits: south.
> get key
You picked up: key
> get key
There is no key here
> inventory
You have: ['key']
> dance now
Unknown command: dance
>
Say something!
> open gate
The gate is locked or too far away
> go south
You are in the hall. Exits: north, east.
> go east
You are in the garden. A locked gate is here. Exits: west.
> open gate
You opened the gate with the key. You win!
```

## 5. Две типичные ловушки

### Ловушка 1: строки — не последовательности для `match`

Мы не случайно вызывали `command.split()`, а не сопоставляли саму строку.
Для образцов-последовательностей `str` (а также `bytes`) **не считаются**
последовательностью, хотя в остальном строка — полноценная последовательность
символов.

In [ ]:
for value in [[1, 2], (3, 4), range(5, 7), "89", {10, 11}]:
    match value:
        case [first, second]:
            print(f"{value!r}: matched, first={first}, second={second}")

        case _:
            print(f"{value!r}: not matched")

Образец-последовательность подходит к спискам, кортежам, `range` и другим
последовательностям, но **не** к строкам и **не** к множествам (у них нет
понятия «позиция»).

### Ловушка 2: имя в образце — это захват, а не сравнение

Голое имя в образце (`NORTH`) **всегда** означает захват: «подойди к чему
угодно и свяжи с этим именем». Оно **не** сравнивается со значением
одноименной переменной!

In [ ]:
NORTH = "north"
direction = "south"

match direction:
    case NORTH:
        print("Going north")  # сработало, хотя direction == "south"!

print(NORTH)  # south — константа тихо перезаписана

Поэтому сравнивать значение с константой, хранящейся в переменной, нужно в
**guard**, а не в самом образце. Либо использовать литерал прямо в образце:
`case "north":`.

In [ ]:
NORTH = "north"
direction = "south"

match direction:
    case direction_name if direction_name == NORTH:
        print("Going north")

    case _:
        print("Not north")  # Not north

print(NORTH)  # north — константа не пострадала

## Итоги: виды образцов

| Образец | Пример | Что делает |
|---|---|---|
| литерал | `"quit"`, `404`, `None` | сравнивает значение с константой (`==`, а `None`/`True`/`False` — через `is`) |
| захват | `target` | подходит ко всему и связывает значение с именем |
| wildcard | `_` | подходит ко всему, ничего не связывает |
| последовательность | `[action, target]`, `["drop", *items]` | проверяет длину и разбирает элементы; `str` не подходит |
| or-образец | `["quit"] \| ["exit"]` | подходит, если подошла любая альтернатива |
| `as`-образец | `("north" \| "south") as direction` | связывает подошедшее подвыражение с именем |
| guard | `case [...] if условие` | дополнительное условие после совпадения образца |